# Inter-annotator agreement

Table 2.

In [ ]:
import sys
sys.path.insert(0, "..")

from embodiment import config as cfg

from collections import Counter

import pandas as pd

df = pd.read_csv(cfg.DOUBLE_CODED)
print(f"{len(df)} rows, annotators {sorted(df['annotator'].unique())}")

## Pair the annotations

In [ ]:
LABEL_NORM = {
    "NO_REL": "no_rel", "CHARACTER_MOVE": "character_move",
    "HAPTIC_MATERIAL": "haptic_material",
    "CHARACTER_FELTNESS": "character_feltness",
    "SIGHT_HUMAN": "sight_human", "BOOK_PARATEXT": "book_paratext",
    "HAPTIC_HUMAN_ANIMAL": "haptic_human_animal",
    "SIGHT_MATERIAL": "sight_material",
    "CHARACTER_STATIC": "character_static",
    "DESCRIPTION_FELTNESS": "description_feltness",
    "HUMAN_COGNITION": "character_cognition",
    "facial_expression": "facial_expr",
    "sents_seg_error": "segmentation_error",
}


def is_multi(val):
    s = str(val).strip()
    return s.startswith("[") and s.endswith("]")


def normalise(val):
    if not isinstance(val, str) or is_multi(val):
        return None
    return LABEL_NORM.get(val.strip(), val.strip().lower())


# Pair on (source_file, sentence_id): sentence_id is an index within a book and
# recurs across books, so keying on it alone conflates sentences.
KEY = ["source_file", "sentence_id"]


def single(d):
    return d[~d["label_raw"].apply(is_multi)].drop_duplicates(subset=KEY,
                                                              keep=False)


pairs = single(df[df["annotator"] == "A"]).merge(
    single(df[df["annotator"] == "B"]), on=KEY, suffixes=("_a", "_b"))
pairs["a"] = pairs["label_raw_a"].apply(normalise)
pairs["b"] = pairs["label_raw_b"].apply(normalise)
pairs = pairs.dropna(subset=["a", "b"])
print(f"single-labelled by both: {len(pairs)}")

# Pairs using a label that no longer exists in the scheme are dropped, so the
# figures describe the eleven-category scheme actually in use.
pairs = pairs[~pairs["a"].isin(cfg.RETIRED) & ~pairs["b"].isin(cfg.RETIRED)]
pairs = pairs.reset_index(drop=True)
print(f"after dropping retired labels: {len(pairs)}")

## Overall

In [ ]:
def kappa(a, b):
    n = len(a)
    po = sum(x == y for x, y in zip(a, b)) / n
    ca, cb = Counter(a), Counter(b)
    pe = sum((ca.get(l, 0) / n) * (cb.get(l, 0) / n) for l in set(ca) | set(cb))
    return 1.0 if pe >= 1 else (po - pe) / (1 - pe)


agreement = (pairs["a"] == pairs["b"]).mean()
overall_kappa = kappa(pairs["a"].tolist(), pairs["b"].tolist())
print(f"n           : {len(pairs)}")
print(f"agreement   : {agreement:.4f}")
print(f"Cohen's k   : {overall_kappa:.4f}")

## Per category

In [ ]:
rows = []
for label in sorted(set(pairs["a"]) | set(pairs["b"])):
    a_is, b_is = pairs["a"] == label, pairs["b"] == label
    n_a, n_b, both = int(a_is.sum()), int(b_is.sum()), int((a_is & b_is).sum())
    rows.append({
        "label": label, "n_a": n_a, "n_b": n_b, "both": both,
        # specific agreement 2a/(n1+n2), and kappa one-vs-rest
        "specific_agr": 2 * both / (n_a + n_b),
        "kappa": kappa(a_is.tolist(), b_is.tolist()),
    })

table2 = pd.DataFrame(rows).sort_values("specific_agr", ascending=False)
table2.loc[len(table2)] = {"label": "Macro mean", "n_a": None, "n_b": None,
                           "both": None,
                           "specific_agr": table2["specific_agr"].mean(),
                           "kappa": table2["kappa"].mean()}
table2.loc[len(table2)] = {"label": "Overall", "n_a": len(pairs),
                           "n_b": len(pairs),
                           "both": int((pairs["a"] == pairs["b"]).sum()),
                           "specific_agr": agreement, "kappa": overall_kappa}
table2 = table2.reset_index(drop=True)
cfg.TABLES.mkdir(exist_ok=True)
table2.to_csv(cfg.TABLES / "table2_agreement.csv", index=False)
table2.round(3)

## Confusion

In [ ]:
labels = sorted(set(pairs["a"]) | set(pairs["b"]))
cm = pd.crosstab(pairs["a"], pairs["b"]).reindex(index=labels, columns=labels,
                                                 fill_value=0)

confusions = [
    {"a": i, "b": j, "total": int(cm.loc[i, j] + cm.loc[j, i])}
    for i in labels for j in labels
    if i < j and cm.loc[i, j] + cm.loc[j, i] > 0
]
confusions = pd.DataFrame(confusions).sort_values("total", ascending=False)

# Confusions of the target (person vs material) within one modality: the axis
# the touch/sight comparison in Section 5 rests on.
swaps = sum(int(cm.loc[a, b]) + int(cm.loc[b, a]) for a, b in
            [("haptic_human_animal", "haptic_material"),
             ("sight_human", "sight_material")])
print(f"disagreements: {int((pairs['a'] != pairs['b']).sum())} / {len(pairs)}")
print(f"within-modality target swaps: {swaps}")
confusions.head(10)